In [30]:
from astropy.table import Table
import pandas as pd
import numpy as np
from pandarallel import pandarallel
from concurrent.futures import ThreadPoolExecutor

In [31]:
# Initialize pandarallel to use all available CPUs (8 threads)
pandarallel.initialize(progress_bar=True, nb_workers=8)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [ ]:

def load_votable_to_dataframe(file_path):

    # Read the VOTable file using astropy
    votable = Table.read(file_path, format='votable')

    # Convert to Pandas DataFrame
    df = votable.to_pandas()

    return df

# Example usage
file_path = "C:\\git_repo\\cool-dwarf_stellar_parameter_inference_from_survey_data\\data\\LAMOST_data\\LAMOST_K0-7_Gaia_joined.vot"
df = load_votable_to_dataframe(file_path)


In [4]:
df.head()

,lamost_f_dwarf_oid,obsid,subclass,gaia_source_id,gaia_dr3_id,Plx,e_Plx,Gmag,BPmag,RPmag
0,2655273,299902169,F9,6944962925824,6944962925824,0.990405,0.019034,14.281499,14.667728,13.732877
1,164040,367412181,F9,6944962925824,6944962925824,0.990405,0.019034,14.281499,14.667728,13.732877
2,2655272,299902168,F6,19211389524736,19211389524736,0.369344,0.041127,15.887926,16.193668,15.417393
3,2658475,300702168,F6,19211389524736,19211389524736,0.369344,0.041127,15.887926,16.193668,15.417393
4,1254921,879302161,F6,19211389524736,19211389524736,0.369344,0.041127,15.887926,16.193668,15.417393


### Sanity check

In [5]:
print(df.shape[0])

2629990


original small LAMOST df (F or G or K or M dwarf)

In [ ]:
def load_votable_to_dataframe(file_path):
    # Read the VOTable file using astropy
    votable = Table.read(file_path, format='votable')

    df = votable.to_pandas()

    return df

file_path = "C:\\git_repo\\cool-dwarf_stellar_parameter_inference_from_survey_data\\data\\LAMOST_data\\LAMOST_K0-7.vot"
df_LAMOST = load_votable_to_dataframe(file_path)
print(df_LAMOST.head())

   COLID_92974 COLID_92992          COLID_93001
0    308709246          F7  1522750113286112640
1    308710002          F2  1520634000078176640
2    308710014          F2  1520589469857274880
3    308710029          F9  1517518259003122560
4    308710039          F9  1517509497269821568


In [7]:
# Check the number of rows in the dataframe
num_rows = df_LAMOST.shape[0]
print("Number of rows:", num_rows)

Number of rows: 2681964


In [8]:
# Check if a specific value is in the 'gaia_dr3_id' column
value_to_check = 1522750113286112640
is_value_present = value_to_check in df['gaia_dr3_id'].values
print(f"Is the value {value_to_check} in 'gaia_dr3_id' column?", is_value_present)

Is the value 1522750113286112640 in 'gaia_dr3_id' column? True


### Clean the Gaia X LAMOST_small data

In [9]:
# Drop rows where Parallax is missing or negative (bad data)
df_clean = df.dropna(subset=['obsid', 'Plx', 'Gmag', 'BPmag', 'RPmag'])
df_clean = df_clean[df_clean['Plx'] > 0]  # Parallax must be positive for distance

# Calculate Distance (in parsecs)
# Since Gaia Plx is in milliarcseconds, so we divide 1000 by Plx rather than 1/Plx
df_clean['distance_gaia_pc'] = 1000 / df_clean['Plx']

# Calculate Absolute G Magnitude (M_G)
# Formula: M = m - 5*log10(d) + 5
#df_clean['abs_Gmag'] = df_clean['Gmag'] - 5 * np.log10(df_clean['distance_gaia_pc']) + 5
#df_clean['abs_Bmag'] = df_clean['BPmag'] - 5 * np.log10(df_clean['distance_gaia_pc']) + 5
#df_clean['abs_Rmag'] = df_clean['RPmag'] - 5 * np.log10(df_clean['distance_pc']) + 5

#print(df_clean[['subclass', 'distance_pc', 'abs_Gmag', 'bp_rp']].head())
print(df_clean.head())

   lamost_f_dwarf_oid      obsid subclass  gaia_source_id     gaia_dr3_id  \
0             2655273  299902169       F9   6944962925824   6944962925824   
1              164040  367412181       F9   6944962925824   6944962925824   
2             2655272  299902168       F6  19211389524736  19211389524736   
3             2658475  300702168       F6  19211389524736  19211389524736   
4             1254921  879302161       F6  19211389524736  19211389524736   

        Plx     e_Plx       Gmag      BPmag      RPmag  distance_gaia_pc  
0  0.990405  0.019034  14.281499  14.667728  13.732877       1009.688335  
1  0.990405  0.019034  14.281499  14.667728  13.732877       1009.688335  
2  0.369344  0.041127  15.887926  16.193668  15.417393       2707.504495  
3  0.369344  0.041127  15.887926  16.193668  15.417393       2707.504495  
4  0.369344  0.041127  15.887926  16.193668  15.417393       2707.504495  


In [10]:
print(df_clean.shape[0])

2579174


In [11]:
df_clean = df_clean.drop(columns=['lamost_f_dwarf_oid', 'e_Plx', 'Plx', 'gaia_dr3_id'])

In [12]:
df_clean.head()

,obsid,subclass,gaia_source_id,Gmag,BPmag,RPmag,distance_gaia_pc
0,299902169,F9,6944962925824,14.281499,14.667728,13.732877,1009.688335
1,367412181,F9,6944962925824,14.281499,14.667728,13.732877,1009.688335
2,299902168,F6,19211389524736,15.887926,16.193668,15.417393,2707.504495
3,300702168,F6,19211389524736,15.887926,16.193668,15.417393,2707.504495
4,879302161,F6,19211389524736,15.887926,16.193668,15.417393,2707.504495


### Cross match df_clean with full LAMOST and LAMOST_vac dataset

In [ ]:
# ...existing code...
import pandas as pd
from astropy.table import Table
from concurrent.futures import ThreadPoolExecutor

# Input Paths
# CHANGE 1: Update the path to your new .vot file
lamost_vot_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\LAMOST_data\LAMOST_K0-7_full.vot"
lamost_fits_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\LAMOST_fits_data\dr8_final_vac_flag_parameter.fits"


In [15]:
# Read the .vot file using astropy
votable = Table.read(lamost_vot_path, format='votable')

# Convert to pandas DataFrame
dataframe = votable.to_pandas()

# Display the first few rows of the DataFrame
dataframe.head(10)

,COLID_92974,COLID_92992,COLID_92996,COLID_92997,COLID_92998,COLID_92999,COLID_93000,COLID_93001,COLID_93011,COLID_93019,COLID_93020
0,332613222,F5,-999.0000,-999.0,-999.000000,-999.000,13.125900,1192528632157238784,6319.36,4.234,-0.471
1,332613226,F0,-999.0000,-999.0,13.614200,-999.000,-999.000000,1192594156178353920,6810.63,4.134,-0.269
2,332613231,F2,-999.0000,-999.0,-999.000000,-999.000,19.079000,1192584535451587840,NaN,NaN,NaN
3,332613238,F0,-999.0000,-999.0,19.004601,-999.000,-999.000000,1192579415850574720,6509.04,4.199,-0.483
4,332613248,F5,-999.0000,-999.0,-999.000000,-999.000,13.509400,1192588280663076352,6331.89,4.069,0.051
5,332614002,F0,-999.0000,-999.0,-999.000000,-999.000,19.325001,1194545892394343424,7070.60,4.065,-0.425
6,332614003,F9,14.5349,-999.0,-999.000000,-999.000,-999.000000,1194472465633433728,NaN,NaN,NaN
7,332614028,F7,-999.0000,-999.0,-999.000000,-999.000,-999.000000,1194121171668509312,6275.14,4.170,0.060
8,332614062,F5,12.7942,-999.0,-999.000000,13.615,-999.000000,1194134709405450880,6275.87,4.252,-0.199
9,332614067,F9,-999.0000,-999.0,17.667601,-999.000,-999.000000,1194232772098696576,5747.25,4.250,-0.010


In [16]:

# Helper functions for reading
# CHANGE 2: Define a function to load VOTables using Astropy
def load_vot(path):
    # Reads the VOTable and converts it immediately to a Pandas DataFrame
    return Table.read(path, format='votable').to_pandas()

def load_fits(path):
    return Table.read(path, format='fits').to_pandas()

# Load datasets in parallel (Exploiting threading for I/O bound tasks)
print("Loading datasets in parallel with ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=8) as executor:
    # CHANGE 3: Submit the VOT loading function instead of the CSV one
    future_vot = executor.submit(load_vot, lamost_vot_path)
    future_fits = executor.submit(load_fits, lamost_fits_path)
    
    df_LAMOST = future_vot.result()
    df_LAMOST_vac = future_fits.result()

print("Datasets loaded successfully.")
print(f"df_LAMOST shape: {df_LAMOST.shape}")
print(f"df_LAMOST_vac shape: {df_LAMOST_vac.shape}")

Loading datasets in parallel with ThreadPoolExecutor...
Datasets loaded successfully.
df_LAMOST shape: (2681964, 11)
df_LAMOST_vac shape: (7109030, 112)


In [17]:
# Rename columns in df_LAMOST
column_mapping = {
    'COLID_92974': 'obsid',
    'COLID_92992': 'subclass',
    'COLID_92996': 'mag_ps_g',
    'COLID_92997': 'mag_ps_r',
    'COLID_92998': 'mag_ps_i',
    'COLID_92999': 'mag_ps_z',
    'COLID_93000': 'mag_ps_y',
    'COLID_93001': 'gaia_source_id',
    'COLID_93011': 'teff',
    'COLID_93019': 'logg',
    'COLID_93020': 'feh'
}

# Apply the renaming
df_LAMOST.rename(columns=column_mapping, inplace=True)

# Print the updated column names for verification
print("Updated column names:", df_LAMOST.columns.tolist())

Updated column names: ['obsid', 'subclass', 'mag_ps_g', 'mag_ps_r', 'mag_ps_i', 'mag_ps_z', 'mag_ps_y', 'gaia_source_id', 'teff', 'logg', 'feh']


In [18]:
# Define columns to keep for df_LAMOST_vac
columns_to_keep_vac = ['obsid', 
                       # all absolute magnitudes
                       'A_GG', 'A_BP', 'A_RP', # Gaia EDR3
                       'A_J', 'A_H', 'A_KS', # 2MASS
                       'A_W1', 'A_W2', # WISE
                       'A_BAP', 'A_VAP', 'A_RAP', #APASS
                       'A_GSD', 'A_RSD', 'A_ISD']  # SDSS

# Define columns to drop for df_LAMOST
irrelevant_columns_lamost = ['spid', 'snrr','snri', 'class','subclass', 'ps_id','gaia_source_id', 'gaia_g_mean_mag',
                             'feh' # all NA in this column
                             ] 

# Define columns to check for missing values
# Rows with NaNs in these columns will be dropped
nan_columns_check_lamost = [] 
nan_columns_check_vac = []

# Apply Cleaning
if irrelevant_columns_lamost:
    df_LAMOST.drop(columns=irrelevant_columns_lamost, inplace=True, errors='ignore')

if columns_to_keep_vac:
    df_LAMOST_vac = df_LAMOST_vac[columns_to_keep_vac]

# Drop rows with specific value (-999) in specified columns
columns_to_check_for_value = ['mag_ps_y', 'mag_ps_z', 'mag_ps_i', 'mag_ps_r', 'mag_ps_g'] # apparent magnitudes
for col in columns_to_check_for_value:
    if col in df_LAMOST.columns:
        df_LAMOST = df_LAMOST[df_LAMOST[col] != -999]

if nan_columns_check_lamost:
    df_LAMOST.dropna(subset=nan_columns_check_lamost, inplace=True)

if nan_columns_check_vac:
    df_LAMOST_vac.dropna(subset=nan_columns_check_vac, inplace=True)

print("Cleaning complete")

Cleaning complete


In [19]:
df_LAMOST_vac.head(10)

,obsid,A_GG,A_BP,A_RP,A_J,A_H,A_KS,A_W1,A_W2,A_BAP,A_VAP,A_RAP,A_GSD,A_RSD,A_ISD
0,105704216.0,3.751406,3.881304,2.902264,2.336192,2.156372,1.663003,2.524816,1.010052,3.043267,3.874843,2.984920,2.919531,3.610125,3.552638
1,814404042.0,4.068060,4.360425,4.942953,3.212042,4.168449,2.917255,3.933477,3.162223,6.281215,4.351768,4.687097,4.706560,4.050163,6.133793
2,309907119.0,3.817540,3.999802,2.873478,2.580539,1.899501,2.074955,2.280157,1.881451,4.485433,3.520332,3.793094,4.381365,3.434602,3.553047
3,564506028.0,4.082883,4.700488,3.546080,2.831862,3.216836,2.827840,2.193428,2.779294,5.195091,4.152187,4.562494,4.246381,4.444151,4.589468
4,557708087.0,1.277164,1.984290,0.647106,0.245124,-0.012795,-0.667526,0.023370,-0.435779,2.089422,1.887725,1.910387,2.135766,1.395873,1.528985
5,228605225.0,4.908501,5.517083,4.153411,4.032901,3.515193,3.371768,3.212000,3.304538,6.058166,4.773209,4.693232,5.233153,4.242326,4.134130
6,507005148.0,4.721043,5.804685,4.395596,4.018856,3.903585,3.934841,3.216208,3.275542,6.903694,5.404795,4.855844,4.739204,4.408143,4.957470
7,756711092.0,3.068360,4.554669,2.173767,1.954524,1.686570,1.184146,1.316306,2.600422,3.668927,3.549033,3.549498,2.940325,3.033015,1.547707
8,564508230.0,2.538508,4.394424,3.679119,2.047662,2.201259,2.897748,1.874365,1.385151,5.182198,3.290172,3.144165,3.549851,2.390765,3.044298
9,414402010.0,4.645375,5.539663,4.799868,3.656878,3.637173,3.818758,2.607846,3.200141,5.749143,4.662692,5.437180,4.404426,4.987878,4.183251


In [20]:
df_LAMOST.head(10)

,obsid,mag_ps_g,mag_ps_r,mag_ps_i,mag_ps_z,mag_ps_y,teff,logg
13,332614229,12.9300,12.7860,12.7770,12.8050,12.7686,7117.89,4.025
24,332615131,13.3974,13.2130,13.0480,12.9720,12.9726,5799.00,4.214
36,332616047,13.5209,13.3170,13.0950,12.9700,12.8631,5694.09,4.388
39,332616098,13.9714,13.5282,13.4460,13.3201,13.2820,5564.10,4.550
41,332616158,13.3296,13.3280,13.1940,13.0187,13.0288,6361.82,4.180
57,332701175,13.9105,13.6496,13.3322,13.5160,13.5392,6694.59,4.257
60,332701244,13.1960,12.7700,12.6000,12.5240,12.4695,5641.83,4.185
65,332702079,13.3833,13.3950,13.3210,13.2025,13.1900,6707.00,3.899
71,332702159,13.9187,13.6490,13.5595,13.5551,13.5490,6158.45,4.276
92,332703179,14.2388,13.8166,13.6969,13.6787,13.6506,5728.74,4.313


In [21]:
print(df_LAMOST.shape[0])

766361


In [22]:
print(df_LAMOST_vac.shape[0])

7109030


### Sanity check on df_LAMOST and df_LAMOST_vac

In [24]:
# Drop rows where any column has a missing value in df_LAMOST and df_LAMOST_vac
print("Dropping rows with missing values in df_LAMOST and df_LAMOST_vac...")

# Drop rows with missing values in df_LAMOST
df_LAMOST_cleaned = df_LAMOST.dropna()
print(f"df_LAMOST: Dropped {df_LAMOST.shape[0] - df_LAMOST_cleaned.shape[0]} rows with missing values.")

# Drop rows with missing values in df_LAMOST_vac
df_LAMOST_vac_cleaned = df_LAMOST_vac.dropna()
print(f"df_LAMOST_vac: Dropped {df_LAMOST_vac.shape[0] - df_LAMOST_vac_cleaned.shape[0]} rows with missing values.")

# Display the shapes of the cleaned dataframes
print("Cleaned DataFrame Shapes:")
print(f"df_LAMOST_cleaned: {df_LAMOST_cleaned.shape}")
print(f"df_LAMOST_vac_cleaned: {df_LAMOST_vac_cleaned.shape}")

Dropping rows with missing values in df_LAMOST and df_LAMOST_vac...
df_LAMOST: Dropped 180018 rows with missing values.
df_LAMOST_vac: Dropped 146 rows with missing values.
Cleaned DataFrame Shapes:
df_LAMOST_cleaned: (586343, 8)
df_LAMOST_vac_cleaned: (7108884, 15)


In [25]:
print("Merging df_clean with df_LAMOST")
# First Inner Join
df_merged_1 = df_clean.merge(df_LAMOST_cleaned, how='inner', on='obsid') 

Merging df_clean with df_LAMOST


In [26]:
print("Merging with df_LAMOST_vac")
# Second inner Join
df_final = df_merged_1.merge(df_LAMOST_vac_cleaned, how='inner', on='obsid')

Merging with df_LAMOST_vac


In [27]:
print(df_final.shape)

(485324, 28)


In [28]:
print(df_final.columns.tolist())

['obsid', 'subclass', 'gaia_source_id', 'Gmag', 'BPmag', 'RPmag', 'distance_gaia_pc', 'mag_ps_g', 'mag_ps_r', 'mag_ps_i', 'mag_ps_z', 'mag_ps_y', 'teff', 'logg', 'A_GG', 'A_BP', 'A_RP', 'A_J', 'A_H', 'A_KS', 'A_W1', 'A_W2', 'A_BAP', 'A_VAP', 'A_RAP', 'A_GSD', 'A_RSD', 'A_ISD']


In [ ]:
output_path = r"C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\Xmatch_result\Xmatch_gaia_LAMOST_LAMOSTvac_F_dwarf.csv"
df_final.to_csv(output_path, index=False)
print(f"df_final saved to {output_path}")

df_final saved to C:\git_repo\cool-dwarf_stellar_parameter_inference_from_survey_data\data\Xmatch_gaia_LAMOST_LAMOSTvac_F_dwarf.csv
